# AssetOpsBench Dataset Analysis and Skill-Prep Workflow

This notebook loads `ibm-research/AssetOpsBench` (`scenarios` config), profiles schema/content, and prepares a semantic-deduplicated dataset for downstream LLM skill generation.

## Workflow overview

1. Dataset loading and schema profiling.
2. Query normalization and entity masking.
3. Baseline semantic grouping and representative selection.
4. Refinement with threshold sweep and merge-safety diagnostics.
5. Optional controlled aggressive merge for additional compression.
6. CSV exports under `data/` for downstream skill-generation prompts.

## Final objective

Produce grouped records where each row contains:
- one representative query,
- its expected answer form (`characteristic_form`),
- supporting note (`note`),
- and member queries belonging to the same semantic intent cluster.

In [11]:
from datasets import load_dataset
import pandas as pd
from collections import Counter
from pprint import pprint

# Load the scenarios configuration
# If this is your first run, this will download/cache from Hugging Face.
ds = load_dataset("ibm-research/AssetOpsBench", "scenarios")

ds

DatasetDict({
    train: Dataset({
        features: ['id', 'type', 'text', 'category', 'deterministic', 'characteristic_form', 'group', 'entity', 'note'],
        num_rows: 152
    })
})

In [12]:
# Inspect split names and row counts
for split_name, split_data in ds.items():
    print(f"{split_name}: {len(split_data)} rows")

train: 152 rows


In [13]:
# Column names and schema-level feature definitions
split = "train" if "train" in ds else list(ds.keys())[0]
print(f"Using split: {split}\n")

print("Columns:")
print(ds[split].column_names)

print("\nFeature schema:")
pprint(ds[split].features)

Using split: train

Columns:
['id', 'type', 'text', 'category', 'deterministic', 'characteristic_form', 'group', 'entity', 'note']

Feature schema:
{'category': Value('string'),
 'characteristic_form': Value('string'),
 'deterministic': Value('bool'),
 'entity': Value('string'),
 'group': Value('string'),
 'id': Value('int64'),
 'note': Value('string'),
 'text': Value('string'),
 'type': Value('string')}


In [14]:
# Convert a split to pandas for easier ad-hoc profiling
# (for very large splits, use a subset instead of full conversion)
df = ds[split].to_pandas()

print("DataFrame shape:", df.shape)
df.head(3)

DataFrame shape: (152, 9)


,id,type,text,category,deterministic,characteristic_form,group,entity,note
0,1,IoT,What IoT sites are available?,Knowledge Query,True,The expected response should be the return val...,retrospective,Site,Source: IoT data operations; Deterministic que...
1,2,IoT,Can you list the IoT sites?,Knowledge Query,True,The expected response should be the return val...,retrospective,Site,Source: IoT data operations; Deterministic que...
2,3,IoT,What assets can be found at the MAIN site?,Knowledge Query,True,The expected response should be the return val...,retrospective,Site,Source: IoT data operations; Deterministic que...


In [15]:
# Column-by-column format inspection
# - pandas dtype
# - Hugging Face feature type
# - missing values
# - Python runtime types seen in sample rows
# - example values

def summarize_column(col, n_examples=3, sample_n=200):
    series = df[col]
    feature = ds[split].features.get(col)

    # Sample to avoid expensive full-column type scans
    sample_series = series.head(sample_n)
    python_types = Counter(type(v).__name__ for v in sample_series if pd.notna(v))

    examples = []
    for v in series.head(n_examples):
        examples.append(v)

    return {
        "column": col,
        "pandas_dtype": str(series.dtype),
        "hf_feature": str(feature),
        "null_count": int(series.isna().sum()),
        "null_fraction": float(series.isna().mean()) if len(series) else 0.0,
        "python_types_in_sample": dict(python_types),
        "example_values": examples,
    }

column_summaries = [summarize_column(c) for c in df.columns]
profile_df = pd.DataFrame(column_summaries)
profile_df

,column,pandas_dtype,hf_feature,null_count,null_fraction,python_types_in_sample,example_values
0,id,int64,Value('int64'),0,0.0,{'int': 152},"[1, 2, 3]"
1,type,str,Value('string'),0,0.0,{'str': 152},"[IoT, IoT, IoT]"
2,text,str,Value('string'),0,0.0,{'str': 152},"[What IoT sites are available?, Can you list t..."
3,category,str,Value('string'),0,0.0,{'str': 152},"[Knowledge Query, Knowledge Query, Knowledge Q..."
4,deterministic,bool,Value('bool'),0,0.0,{'bool': 152},"[True, True, True]"
5,characteristic_form,str,Value('string'),0,0.0,{'str': 152},[The expected response should be the return va...
6,group,str,Value('string'),0,0.0,{'str': 152},"[retrospective, retrospective, retrospective]"
7,entity,str,Value('string'),0,0.0,{'str': 152},"[Site, Site, Site]"
8,note,str,Value('string'),0,0.0,{'str': 152},[Source: IoT data operations; Deterministic qu...


## Skill-Generation Dataset Preparation Pipeline

Goal: prepare AssetOpsBench queries into representative semantic groups that can be sent to an LLM for skill generation.

Per-group payload requirements for downstream prompting:
- representative query,
- expected answer form (`characteristic_form`),
- source `note`,
- member queries captured by the group.

Pipeline stages in this notebook:
1. Normalize text.
2. Apply entity masking.
3. Build semantic groups and select representatives.
4. Refine thresholds and audit merge safety.
5. Optionally run an additional controlled aggressive merge.
6. Export final CSV artifacts under `data/`.

In [16]:
# -------------------------------------------------------------------
# Dataset preparation pipeline for skill-generation-ready query groups
# -------------------------------------------------------------------
# Goal:
# 1) Normalize + mask entity-specific tokens in queries
# 2) Group semantically equivalent queries
# 3) Pick one representative query per group
# 4) Preserve expected answer + note for downstream LLM skill generation

import re
from pathlib import Path
import numpy as np

# Keep a focused working frame for this pipeline
df_work = df.copy()
required_cols = ["id", "text", "characteristic_form", "note", "type", "category", "group", "entity"]
missing_cols = [c for c in required_cols if c not in df_work.columns]
if missing_cols:
    raise KeyError(f"Missing required columns for prep pipeline: {missing_cols}")

print("=" * 80)
print("STEP 0 - Initial dataset snapshot")
print("=" * 80)
print(f"Input rows: {len(df_work)}")
print(f"Columns used: {required_cols}")
print("\nSample rows (id, text, entity):")
print(df_work[["id", "text", "entity"]].head(8).to_string(index=False))

# ----------------------
# STEP 1: Text normalize
# ----------------------
def normalize_query(text: str) -> str:
    t = str(text).strip().lower()
    t = re.sub(r"[^\w\s]", " ", t)  # remove punctuation
    t = re.sub(r"\s+", " ", t).strip()
    return t

df_work["query_normalized"] = df_work["text"].apply(normalize_query)

exact_unique_before = df_work["text"].nunique()
exact_unique_after_norm = df_work["query_normalized"].nunique()

print("\n" + "=" * 80)
print("STEP 1 - Normalization")
print("=" * 80)
print(f"Unique raw query strings: {exact_unique_before}")
print(f"Unique normalized query strings: {exact_unique_after_norm}")
print("\nNormalization examples:")
print(df_work[["text", "query_normalized"]].head(8).to_string(index=False))

# -------------------------------------------
# STEP 2: Entity masking (hybrid rules)
# -------------------------------------------
# Notes:
# - We preserve operation words (list, count, show, etc.)
# - We mask IDs/timestamps/numeric literals and some explicit entity mentions

ID_PAT = re.compile(r"\b(?:[A-Za-z]{1,5}-\d{2,}|\d{5,}|[0-9a-fA-F]{8}-[0-9a-fA-F-]{27,})\b")
DATE_PAT = re.compile(r"\b\d{4}-\d{2}-\d{2}\b")
REL_TIME_PAT = re.compile(r"\b(?:last|past)\s+\d+\s+(?:day|days|hour|hours|week|weeks|month|months)\b", re.IGNORECASE)
NUM_PAT = re.compile(r"(?<![A-Za-z])\d+(?:\.\d+)?(?![A-Za-z])")

# Small domain-aware phrase patterns; tune these if needed.
SITE_NAME_PAT = re.compile(r"\b(?:at|in|from)\s+the\s+[A-Za-z0-9_-]+\s+site\b", re.IGNORECASE)
ASSET_NAME_PAT = re.compile(r"\b(?:asset|equipment|device)\s+[A-Za-z0-9_-]+\b", re.IGNORECASE)

def mask_entities(text: str) -> str:
    t = str(text)

    # Domain phrase masking first
    t = SITE_NAME_PAT.sub(" at the <SITE> site", t)
    t = ASSET_NAME_PAT.sub("<ASSET>", t)

    # Generic masking
    t = ID_PAT.sub("<ID>", t)
    t = DATE_PAT.sub("<DATE>", t)
    t = REL_TIME_PAT.sub("<TIME_RANGE>", t)
    t = NUM_PAT.sub("<NUM>", t)

    # normalize spacing + lowercase for embedding consistency
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t

df_work["query_masked"] = df_work["text"].apply(mask_entities)

print("\n" + "=" * 80)
print("STEP 2 - Entity masking")
print("=" * 80)
print(f"Unique masked query strings: {df_work['query_masked'].nunique()}")
print("\nMasking examples (raw -> masked):")
print(df_work[["text", "query_masked"]].head(12).to_string(index=False))

# Keep this table for the next step
prep_preview_cols = ["id", "text", "query_normalized", "query_masked", "entity", "type", "category", "group", "characteristic_form", "note"]
print("\nPrepared frame columns for clustering:")
print(prep_preview_cols)
print(f"Prepared frame shape: {df_work[prep_preview_cols].shape}")

STEP 0 - Initial dataset snapshot
Input rows: 152
Columns used: ['id', 'text', 'characteristic_form', 'note', 'type', 'category', 'group', 'entity']

Sample rows (id, text, entity):
 id                                                                                     text    entity
  1                                                            What IoT sites are available?      Site
  2                                                              Can you list the IoT sites?      Site
  3                                               What assets can be found at the MAIN site?      Site
  4                                           Which assets are located at the MAIN facility? Equipment
  5                                Retrieve metadata for Chiller 6 located at the MAIN site.   Chiller
  6                                    Get the asset details for Chiller 9 at the MAIN site.   Chiller
  7                                Download the metadata for Chiller 3 at the MAIN facility.   Ch

In [17]:
# Focused analysis of the `note` column format
if "note" not in df.columns:
    raise KeyError("No `note` column found in this split.")

note_series = df["note"]
print("Rows:", len(note_series))
print("Null notes:", int(note_series.isna().sum()))
print("Non-null notes:", int(note_series.notna().sum()))

# Normalize to strings for text pattern checks
note_text = note_series.dropna().astype(str)
print("Unique non-null notes:", note_text.nunique())

note_text.head(5)

Rows: 152
Null notes: 0
Non-null notes: 152
Unique non-null notes: 17


0    Source: IoT data operations; Deterministic que...
1    Source: IoT data operations; Deterministic que...
2    Source: IoT data operations; Deterministic que...
3    Source: IoT data operations; Deterministic que...
4    Source: IoT data operations; Deterministic que...
Name: note, dtype: str

## Baseline Semantic Grouping and Representative Selection

This section builds the first semantic grouping baseline.

Method summary:
- Compute masked-query similarities.
- Build connected components within metadata blocks (`type`, `entity`, `category`).
- Select one medoid representative per component.
- Carry forward representative query, expected answer (`characteristic_form`), and `note`, plus full member lists.

Output:
- Initial representative dataset written to `data/assetopsbench_skill_prep_representative_queries.csv`.

In [ ]:
# -------------------------------------------------------------------
# STEP 3+: Semantic grouping, representative selection, CSV export
# -------------------------------------------------------------------
from sklearn.metrics.pairwise import cosine_similarity

# Try sentence-transformers first, fallback to TF-IDF if unavailable.
# This keeps the notebook runnable across environments.
def compute_embeddings(texts):
    try:
        from sentence_transformers import SentenceTransformer
        model_name = "BAAI/bge-base-en-v1.5"
        model = SentenceTransformer(model_name)
        emb = model.encode(
            texts,
            normalize_embeddings=True,
            batch_size=64,
            show_progress_bar=True,
        )
        return np.asarray(emb), f"sentence-transformers ({model_name})"
    except Exception as e:
        print("[WARN] sentence-transformers unavailable; falling back to TF-IDF.")
        print(f"[WARN] reason: {e}")
        from sklearn.feature_extraction.text import TfidfVectorizer
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform(texts)
        # L2 normalized by default in TfidfVectorizer, compatible with cosine
        return X, "TF-IDF (fallback)"


def connected_components_from_similarity(sim_matrix, threshold):
    """Return connected components as lists of row indices."""
    n = sim_matrix.shape[0]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= threshold:
                union(i, j)

    groups = {}
    for i in range(n):
        r = find(i)
        groups.setdefault(r, []).append(i)

    return list(groups.values())


# Block by high-signal metadata to reduce false merges
# (same task family + same entity + same category)
block_cols = ["type", "entity", "category"]
threshold = 0.88

print("\n" + "=" * 80)
print("STEP 3 - Semantic grouping setup")
print("=" * 80)
print(f"Blocking columns: {block_cols}")
print(f"Similarity threshold: {threshold}")

blocks = list(df_work.groupby(block_cols, dropna=False).groups.items())
print(f"Total blocks: {len(blocks)}")
print("Sample block keys with size:")
for block_key, idxs in blocks[:8]:
    print(f"  {block_key}: {len(idxs)} rows")

cluster_rows = []
cluster_id = 0

for block_key, idxs in blocks:
    sub = df_work.loc[list(idxs)].copy().reset_index(drop=False)
    # index column now points to original df_work index

    texts = sub["query_masked"].tolist()

    if len(sub) == 1:
        cluster_id += 1
        row = sub.iloc[0]
        cluster_rows.append({
            "cluster_id": cluster_id,
            "block_key": str(block_key),
            "rep_df_index": int(row["index"]),
            "member_df_indices": [int(row["index"])],
            "cluster_size": 1,
            "embedding_backend": "N/A (singleton)",
        })
        continue

    emb, backend = compute_embeddings(texts)
    sim = cosine_similarity(emb)
    components = connected_components_from_similarity(sim, threshold=threshold)

    for comp in components:
        comp_sub = sub.iloc[comp]

        if len(comp) == 1:
            rep_local_pos = comp[0]
        else:
            # Medoid: highest mean similarity to others in same component
            comp_sim = sim[np.ix_(comp, comp)]
            local_best = int(np.argmax(comp_sim.mean(axis=1)))
            rep_local_pos = comp[local_best]

        rep_row = sub.iloc[rep_local_pos]

        cluster_id += 1
        cluster_rows.append({
            "cluster_id": cluster_id,
            "block_key": str(block_key),
            "rep_df_index": int(rep_row["index"]),
            "member_df_indices": [int(x) for x in comp_sub["index"].tolist()],
            "cluster_size": int(len(comp_sub)),
            "embedding_backend": backend,
        })

clusters_df = pd.DataFrame(cluster_rows)

print("\n" + "=" * 80)
print("STEP 4 - Cluster stats")
print("=" * 80)
print(f"Total clusters: {len(clusters_df)}")
print(f"Original rows: {len(df_work)}")
print(f"Compression ratio: {len(df_work) / max(len(clusters_df), 1):.2f}x")
print("\nCluster size distribution:")
print(clusters_df["cluster_size"].value_counts().sort_index().to_string())

# Expand cluster members into query lists for downstream skill generation
id_to_row = df_work.reset_index(drop=False).set_index("index")

def collect_member_queries(member_indices):
    rows = id_to_row.loc[member_indices]
    return rows["text"].tolist()


def collect_member_expected(member_indices):
    rows = id_to_row.loc[member_indices]
    return rows["characteristic_form"].tolist()


def collect_member_notes(member_indices):
    rows = id_to_row.loc[member_indices]
    return rows["note"].tolist()

clusters_df["member_queries"] = clusters_df["member_df_indices"].apply(collect_member_queries)
clusters_df["member_expected_answers"] = clusters_df["member_df_indices"].apply(collect_member_expected)
clusters_df["member_notes"] = clusters_df["member_df_indices"].apply(collect_member_notes)

# Build representative dataset
rep_rows = []
for _, c in clusters_df.iterrows():
    rep = id_to_row.loc[c["rep_df_index"]]
    rep_rows.append({
        "cluster_id": int(c["cluster_id"]),
        "type": rep["type"],
        "entity": rep["entity"],
        "category": rep["category"],
        "group": rep["group"],
        "representative_query": rep["text"],
        "representative_query_masked": rep["query_masked"],
        "representative_expected_answer": rep["characteristic_form"],
        "representative_note": rep["note"],
        "cluster_size": int(c["cluster_size"]),
        "member_queries": c["member_queries"],
        "member_expected_answers": c["member_expected_answers"],
        "member_notes": c["member_notes"],
        "embedding_backend": c["embedding_backend"],
        "block_key": c["block_key"],
    })

prepared_df = pd.DataFrame(rep_rows).sort_values(["type", "entity", "cluster_id"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("STEP 5 - Prepared representative dataset")
print("=" * 80)
print(f"Prepared rows (one per semantic group): {len(prepared_df)}")
print("\nPreview columns:")
preview_cols = [
    "cluster_id", "type", "entity", "representative_query", "cluster_size"
]
print(prepared_df[preview_cols].head(12).to_string(index=False))

print("\nExample of grouped member queries for first few clusters:")
for _, row in prepared_df.head(3).iterrows():
    print("-" * 80)
    print(f"cluster_id={row['cluster_id']} | entity={row['entity']} | size={row['cluster_size']}")
    print(f"representative_query: {row['representative_query']}")
    for q in row["member_queries"][:5]:
        print(f"  * {q}")

# ------------------------------
# STEP 6: Write output to CSV
# ------------------------------
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)
out_file = out_dir / "assetopsbench_skill_prep_representative_queries.csv"

prepared_df.to_csv(out_file, index=False)

print("\n" + "=" * 80)
print("STEP 6 - Export")
print("=" * 80)
print(f"Wrote CSV: {out_file.resolve()}")
print(f"Rows written: {len(prepared_df)}")
print(f"Columns written: {list(prepared_df.columns)}")

prepared_df.head(10)


STEP 3 - Semantic grouping setup
Blocking columns: ['type', 'entity', 'category']
Similarity threshold: 0.88
Total blocks: 18
Sample block keys with size:
  ('FMSA', 'Chiller', 'Knowledge Query'): 18 rows
  ('FMSA', 'WindTurbine', 'Knowledge Query'): 2 rows
  ('IoT', 'AHU', 'Knowledge Query'): 6 rows
  ('IoT', 'Chiller', 'Data Query'): 8 rows
  ('IoT', 'Chiller', 'Knowledge Query'): 2 rows
  ('IoT', 'Equipment', 'Knowledge Query'): 1 rows
  ('IoT', 'Site', 'Knowledge Query'): 3 rows
  ('TSFM', 'Chiller', 'Anomaly Detection Query'): 1 rows


Loading weights: 100%|██████████| 199/199 [00:00<00:00, 6364.31it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 18482.84it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
Loading weights: 100%|██████████| 199/199 [00:00<00:00, 24684.05it/s]
BertModel LOAD REPORT from: BAAI/bge-base-en-v1.5
Key                     | Status     |  | 
------------------------+------------+--+-
embeddings.position_ids | UNEXPECTED |  | 

Notes:
- UN

## Refinement Pass: Threshold Tuning and Safety Audit

This section improves baseline semantic grouping with a conservative masking strategy and explicit threshold tuning.

What it adds:
- Conservative masking profile that masks IDs/date/time expressions but avoids over-masking generic numeric parameters.
- Threshold sweep (`0.82` to `0.92`) with metrics printed for each candidate.
- Heuristic suspicious-merge diagnostics to flag potentially unsafe merges.

Primary outputs:
- `prepared_df_refined` in-memory table.
- CSV exports under `data/`:
  - `assetopsbench_skill_prep_representative_queries.csv`
  - `assetopsbench_skill_prep_representative_queries_refined.csv`
  - `assetopsbench_skill_prep_threshold_eval.csv`

In [ ]:
# -------------------------------------------------------------------
# Refinement pass: threshold sweep + conservative masking + audit
# -------------------------------------------------------------------
from difflib import SequenceMatcher
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\n" + "=" * 80)
print("REFINEMENT - Start")
print("=" * 80)
print("Goal: choose a robust semantic grouping setup for downstream skill generation.")

# Conservative masked variant:
# - Keep generic numbers so parameter-sensitive queries can remain separable when needed.
# - Still mask IDs and date/time literals.
ID_PAT_REF = re.compile(r"\b(?:[A-Za-z]{1,5}-\d{2,}|\d{5,}|[0-9a-fA-F]{8}-[0-9a-fA-F-]{27,})\b")
DATE_PAT_REF = re.compile(r"\b\d{4}-\d{2}-\d{2}(?:[T\s]\d{2}:\d{2}:\d{2})?\b")
REL_TIME_PAT_REF = re.compile(r"\b(?:last|past)\s+\d+\s+(?:day|days|hour|hours|week|weeks|month|months)\b", re.IGNORECASE)
SITE_NAME_PAT_REF = re.compile(r"\b(?:at|in|from)\s+the\s+[A-Za-z0-9_-]+\s+site\b", re.IGNORECASE)


def mask_entities_refined(text: str) -> str:
    t = str(text)
    t = SITE_NAME_PAT_REF.sub(" at the <SITE> site", t)
    t = ID_PAT_REF.sub("<ID>", t)
    t = DATE_PAT_REF.sub("<DATE>", t)
    t = REL_TIME_PAT_REF.sub("<TIME_RANGE>", t)
    t = re.sub(r"\s+", " ", t).strip().lower()
    return t


df_ref = df_work.copy()
df_ref["query_masked_refined"] = df_ref["text"].apply(mask_entities_refined)

print("\nMasking comparison stats:")
print(f"  unique raw queries:           {df_ref['text'].nunique()}")
print(f"  unique previous masked:       {df_ref['query_masked'].nunique()}")
print(f"  unique refined masked:        {df_ref['query_masked_refined'].nunique()}")
print("\nExamples (raw -> refined masked):")
print(df_ref[["text", "query_masked_refined"]].head(10).to_string(index=False))

# We keep same semantic block to reduce false positive merges across task families.
ref_block_cols = ["type", "entity", "category"]
ref_blocks = list(df_ref.groupby(ref_block_cols, dropna=False).groups.items())


def components_from_sim(sim_matrix, threshold):
    n = sim_matrix.shape[0]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= threshold:
                union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def run_grouping_once(dataframe, text_col, threshold):
    vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)

    rows = []
    cid = 0

    for block_key, idxs in ref_blocks:
        sub = dataframe.loc[list(idxs)].copy().reset_index(drop=False)

        if len(sub) == 1:
            cid += 1
            r = sub.iloc[0]
            rows.append({
                "cluster_id": cid,
                "block_key": str(block_key),
                "rep_df_index": int(r["index"]),
                "member_df_indices": [int(r["index"])],
                "cluster_size": 1,
            })
            continue

        X = vec.fit_transform(sub[text_col].tolist())
        sim = cosine_similarity(X)
        comps = components_from_sim(sim, threshold)

        for comp in comps:
            comp_sub = sub.iloc[comp]
            if len(comp) == 1:
                rep_local = comp[0]
            else:
                csim = sim[np.ix_(comp, comp)]
                rep_local = comp[int(np.argmax(csim.mean(axis=1)))]

            rep_row = sub.iloc[rep_local]
            cid += 1
            rows.append({
                "cluster_id": cid,
                "block_key": str(block_key),
                "rep_df_index": int(rep_row["index"]),
                "member_df_indices": [int(x) for x in comp_sub["index"].tolist()],
                "cluster_size": int(len(comp_sub)),
            })

    out = pd.DataFrame(rows)
    return out


def suspicious_merge_count(cluster_df, base_df):
    # Heuristic only: low lexical similarity inside merged cluster may indicate risk.
    c = 0
    examples = []
    idx_df = base_df.reset_index(drop=False).set_index("index")
    for _, row in cluster_df.iterrows():
        if row["cluster_size"] <= 1:
            continue
        qs = idx_df.loc[row["member_df_indices"], "text"].tolist()
        ratios = []
        for i in range(len(qs)):
            for j in range(i + 1, len(qs)):
                ratios.append(SequenceMatcher(None, qs[i].lower(), qs[j].lower()).ratio())
        avg_ratio = float(np.mean(ratios)) if ratios else 1.0
        if avg_ratio < 0.33:
            c += 1
            if len(examples) < 5:
                examples.append((row["cluster_id"], avg_ratio, qs))
    return c, examples


# Threshold sweep
candidate_thresholds = [0.82, 0.85, 0.88, 0.90, 0.92]
records = []

for thr in candidate_thresholds:
    cdf = run_grouping_once(df_ref, text_col="query_masked_refined", threshold=thr)
    suspicious_n, _ = suspicious_merge_count(cdf, df_ref)
    records.append({
        "threshold": thr,
        "clusters": int(len(cdf)),
        "compression_ratio": float(len(df_ref) / max(len(cdf), 1)),
        "merged_groups": int((cdf["cluster_size"] > 1).sum()),
        "max_cluster_size": int(cdf["cluster_size"].max()),
        "suspicious_merge_groups": int(suspicious_n),
    })

threshold_eval_df = pd.DataFrame(records).sort_values(["suspicious_merge_groups", "clusters"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("REFINEMENT - Threshold sweep")
print("=" * 80)
print(threshold_eval_df.to_string(index=False))

# Selection strategy:
# 1) Min suspicious merges
# 2) Then maximize compression (i.e., fewer clusters)
best = threshold_eval_df.iloc[0]
best_threshold = float(best["threshold"])
print(f"\nChosen threshold: {best_threshold}")

clusters_ref = run_grouping_once(df_ref, text_col="query_masked_refined", threshold=best_threshold)
suspicious_n, suspicious_examples = suspicious_merge_count(clusters_ref, df_ref)

print("\n" + "=" * 80)
print("REFINEMENT - Final cluster diagnostics")
print("=" * 80)
print(f"Final clusters: {len(clusters_ref)}")
print(f"Final compression ratio: {len(df_ref)/max(len(clusters_ref),1):.3f}x")
print("Cluster size distribution:")
print(clusters_ref["cluster_size"].value_counts().sort_index().to_string())
print(f"Heuristic suspicious merged groups: {suspicious_n}")

if suspicious_examples:
    print("\nSuspicious merge examples (for manual inspection):")
    for cid, score, qs in suspicious_examples:
        print("-" * 80)
        print(f"cluster_id={cid}, avg_pairwise_lexical_ratio={score:.3f}")
        for q in qs:
            print(f"  * {q}")

# Build final representative dataset
id_df = df_ref.reset_index(drop=False).set_index("index")
final_rows = []
for _, c in clusters_ref.iterrows():
    rep = id_df.loc[c["rep_df_index"]]
    member_rows = id_df.loc[c["member_df_indices"]]
    final_rows.append({
        "cluster_id": int(c["cluster_id"]),
        "type": rep["type"],
        "entity": rep["entity"],
        "category": rep["category"],
        "group": rep["group"],
        "representative_query": rep["text"],
        "representative_query_masked": rep["query_masked_refined"],
        "representative_expected_answer": rep["characteristic_form"],
        "representative_note": rep["note"],
        "cluster_size": int(c["cluster_size"]),
        "member_queries": member_rows["text"].tolist(),
        "member_expected_answers": member_rows["characteristic_form"].tolist(),
        "member_notes": member_rows["note"].tolist(),
        "threshold_used": best_threshold,
        "masking_profile": "refined_keep_numeric_mask_dates_ids",
        "block_key": c["block_key"],
    })

prepared_df_refined = pd.DataFrame(final_rows).sort_values(["type", "entity", "cluster_id"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("REFINEMENT - Prepared dataset preview")
print("=" * 80)
print(f"Rows: {len(prepared_df_refined)}")
print(prepared_df_refined[["cluster_id", "type", "entity", "representative_query", "cluster_size"]].head(12).to_string(index=False))

print("\nFirst 3 groups with member queries:")
for _, row in prepared_df_refined.head(3).iterrows():
    print("-" * 80)
    print(f"cluster_id={row['cluster_id']} | size={row['cluster_size']} | entity={row['entity']}")
    print(f"representative_query: {row['representative_query']}")
    for q in row["member_queries"][:6]:
        print(f"  * {q}")

# Export (overwrite prior file with refined output + write versioned file)
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)
out_file_main = out_dir / "assetopsbench_skill_prep_representative_queries.csv"
out_file_refined = out_dir / "assetopsbench_skill_prep_representative_queries_refined.csv"

prepared_df_refined.to_csv(out_file_main, index=False)
prepared_df_refined.to_csv(out_file_refined, index=False)

# Also export threshold report for traceability
thr_report = out_dir / "assetopsbench_skill_prep_threshold_eval.csv"
threshold_eval_df.to_csv(thr_report, index=False)

print("\n" + "=" * 80)
print("REFINEMENT - Export complete")
print("=" * 80)
print(f"Main CSV: {out_file_main.resolve()}")
print(f"Versioned CSV: {out_file_refined.resolve()}")
print(f"Threshold report: {thr_report.resolve()}")
print(f"Rows written: {len(prepared_df_refined)}")
print(f"Columns: {list(prepared_df_refined.columns)}")

prepared_df_refined.head(10)

## Aggressive Merge (V1, Exploratory)

This section is an exploratory template-collapse attempt.

Notes:
- It uses intent/action signatures and template text keys.
- It is intentionally retained for comparison and reproducibility.
- In later testing, V2 provides safer behavior and clearer controls for parameter-sensitive requests.

Recommendation:
- Prefer the V2 section for final dataset export decisions.

In [ ]:
# -------------------------------------------------------------------
# Aggressive pass: template-intent collapsing with safety guards
# -------------------------------------------------------------------
# Why this pass?
# - The refined pass is conservative and may leave obvious template paraphrases unmerged.
# - Here we add a template-intent signature layer to merge phrasing variants more aggressively,
#   while preserving parameter-sensitive requests.

import ast

print("\n" + "=" * 80)
print("AGGRESSIVE PASS - Start")
print("=" * 80)

# Reuse refined table if available; otherwise rebuild from df_ref + clusters_ref
if "prepared_df_refined" not in globals():
    raise RuntimeError("Run the refinement cell first to create `prepared_df_refined`.")

base_df = prepared_df_refined.copy()
print(f"Input groups from refined pass: {len(base_df)}")

# -----------------------------
# 1) Build intent signatures
# -----------------------------
ACTION_PATTERNS = [
    (r"\b(list|show|what|which)\b", "ASK_LIST"),
    (r"\b(count|how many)\b", "ASK_COUNT"),
    (r"\b(forecast|predict|probability)\b", "ASK_PREDICT"),
    (r"\b(anomal|detect)\b", "ASK_ANOMALY"),
    (r"\b(failure mode)\b", "ASK_FAILURE_MODE"),
    (r"\b(work order|workorder)\b", "ASK_WORKORDER"),
    (r"\b(create|open|generate)\b", "ASK_CREATE"),
]

PARAM_SENSITIVE_PATTERNS = [
    r"\bcontext length\b",
    r"\btop\s+\d+\b",
    r"\bthreshold\b",
    r"\bwindow\b",
    r"\bforecast\b",
    r"\bpredict\b",
]

def to_list(v):
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        try:
            parsed = ast.literal_eval(v)
            if isinstance(parsed, list):
                return parsed
            return [v]
        except Exception:
            return [v]
    return [v]


def detect_action(query: str) -> str:
    q = str(query).lower()
    tags = [tag for pat, tag in ACTION_PATTERNS if re.search(pat, q)]
    if not tags:
        return "ASK_OTHER"
    # Keep deterministic ordering to make signatures stable
    return "+".join(sorted(set(tags)))


def is_param_sensitive(query: str) -> bool:
    q = str(query).lower()
    return any(re.search(p, q) for p in PARAM_SENSITIVE_PATTERNS)


def stable_template_text(query: str) -> str:
    q = str(query).lower()
    # normalize obvious variable segments
    q = re.sub(r"\bchiller\s*\d+\b", "chiller <NUM>", q)
    q = re.sub(r"\bsite\s*[a-z0-9_-]+\b", "site <SITE>", q)
    q = re.sub(r"\bequipment\s*[a-z0-9_-]+\b", "equipment <ID>", q)
    q = re.sub(r"\bfrom\s+<date>\s+to\s+<date>\b", "from <DATE_RANGE>", q)
    q = re.sub(r"\s+", " ", q).strip()
    return q

base_df["action_signature"] = base_df["representative_query"].apply(detect_action)
base_df["param_sensitive"] = base_df["representative_query"].apply(is_param_sensitive)
base_df["template_text"] = base_df["representative_query_masked"].apply(stable_template_text)

print("\nSignature preview:")
print(
    base_df[["cluster_id", "type", "entity", "action_signature", "param_sensitive", "representative_query"]]
    .head(12)
    .to_string(index=False)
)

# ---------------------------------------
# 2) Merge refined groups into meta-groups
# ---------------------------------------
# Merge key is intentionally strict on metadata and action.
# For parameter-sensitive clusters, keep each group isolated by original cluster_id.
def meta_key(row):
    if row["param_sensitive"]:
        return (
            row["type"], row["entity"], row["category"], row["group"],
            row["action_signature"], "PARAM_LOCK", row["cluster_id"]
        )
    return (
        row["type"], row["entity"], row["category"], row["group"],
        row["action_signature"], row["template_text"]
    )

base_df["meta_key"] = base_df.apply(meta_key, axis=1)

meta_rows = []
meta_id = 0

for mk, sub in base_df.groupby("meta_key", dropna=False):
    meta_id += 1
    sub = sub.copy()

    # representative meta-row: largest supporting set; tie-break by shortest query
    sub = sub.sort_values(["cluster_size", "representative_query"], ascending=[False, True])
    rep = sub.iloc[0]

    merged_queries = []
    merged_expected = []
    merged_notes = []

    for _, r in sub.iterrows():
        merged_queries.extend(to_list(r["member_queries"]))
        merged_expected.extend(to_list(r["member_expected_answers"]))
        merged_notes.extend(to_list(r["member_notes"]))

    # de-dup but preserve order
    def dedup_keep_order(items):
        seen = set()
        out = []
        for x in items:
            k = str(x)
            if k not in seen:
                seen.add(k)
                out.append(x)
        return out

    merged_queries = dedup_keep_order(merged_queries)
    merged_expected = dedup_keep_order(merged_expected)
    merged_notes = dedup_keep_order(merged_notes)

    meta_rows.append({
        "meta_cluster_id": meta_id,
        "type": rep["type"],
        "entity": rep["entity"],
        "category": rep["category"],
        "group": rep["group"],
        "representative_query": rep["representative_query"],
        "representative_query_masked": rep["representative_query_masked"],
        "representative_expected_answer": rep["representative_expected_answer"],
        "representative_note": rep["representative_note"],
        "source_refined_cluster_ids": sub["cluster_id"].tolist(),
        "source_refined_cluster_count": int(len(sub)),
        "cluster_size": int(len(merged_queries)),
        "member_queries": merged_queries,
        "member_expected_answers": merged_expected,
        "member_notes": merged_notes,
        "action_signature": rep["action_signature"],
        "param_sensitive": bool(rep["param_sensitive"]),
        "meta_key": str(mk),
        "merge_profile": "aggressive_template_collapse_v1",
    })

prepared_df_aggressive = pd.DataFrame(meta_rows).sort_values(["type", "entity", "meta_cluster_id"]).reset_index(drop=True)

print("\n" + "=" * 80)
print("AGGRESSIVE PASS - Results")
print("=" * 80)
print(f"Refined groups:    {len(prepared_df_refined)}")
print(f"Aggressive groups: {len(prepared_df_aggressive)}")
print(f"Extra compression: {len(prepared_df_refined) / max(len(prepared_df_aggressive), 1):.3f}x over refined")

print("\nAggressive cluster size distribution:")
print(prepared_df_aggressive["cluster_size"].value_counts().sort_index().to_string())

print("\nTop merged meta-groups (largest first):")
largest = prepared_df_aggressive.sort_values("cluster_size", ascending=False).head(8)
print(largest[["meta_cluster_id", "type", "entity", "cluster_size", "action_signature", "param_sensitive", "representative_query"]].to_string(index=False))

print("\nExamples of meta-group members (first 3 groups):")
for _, row in prepared_df_aggressive.head(3).iterrows():
    print("-" * 80)
    print(f"meta_cluster_id={row['meta_cluster_id']} | size={row['cluster_size']} | action={row['action_signature']} | param_sensitive={row['param_sensitive']}")
    print(f"representative_query: {row['representative_query']}")
    for q in row["member_queries"][:8]:
        print(f"  * {q}")

# ------------------------------
# 3) Export aggressive artifacts
# ------------------------------
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)

aggr_main = out_dir / "assetopsbench_skill_prep_representative_queries_aggressive.csv"
aggr_for_skills = out_dir / "assetopsbench_skill_prep_for_skillgen_aggressive.csv"

prepared_df_aggressive.to_csv(aggr_main, index=False)

# Optional compact view for immediate LLM skill generation payloads
skillgen_cols = [
    "meta_cluster_id", "type", "entity", "category", "group",
    "representative_query", "representative_expected_answer", "representative_note",
    "cluster_size", "member_queries", "member_expected_answers", "member_notes",
    "action_signature", "param_sensitive", "merge_profile"
]
prepared_df_aggressive[skillgen_cols].to_csv(aggr_for_skills, index=False)

print("\n" + "=" * 80)
print("AGGRESSIVE PASS - Export")
print("=" * 80)
print(f"Aggressive full CSV: {aggr_main.resolve()}")
print(f"Skillgen CSV:        {aggr_for_skills.resolve()}")
print(f"Rows written:        {len(prepared_df_aggressive)}")
print(f"Columns written:     {list(prepared_df_aggressive.columns)}")

prepared_df_aggressive.head(10)

## Controlled Aggressive Merge (V2)

This section performs a second-stage merge on top of the refined clusters to catch a few additional high-confidence template duplicates.

Design choices:
- Merge only within strict blocks: `type`, `entity`, `category`, `group`, inferred `action_signature`, and `param_sensitive` flag.
- Keep parameter-sensitive requests isolated (for example, context-length- or prediction-parameter-specific queries).
- Use a threshold sweep to evaluate candidate second-stage similarity thresholds and choose the strongest compression setting from the sweep.

Outputs:
- `prepared_df_aggressive_v2` in-memory table.
- CSV exports under `data/`:
  - `assetopsbench_skill_prep_representative_queries_aggressive_v2.csv`
  - `assetopsbench_skill_prep_for_skillgen_aggressive_v2.csv`
  - `assetopsbench_skill_prep_aggressive_v2_sweep.csv`

In [ ]:
# -------------------------------------------------------------------
# Aggressive pass v2: controlled merge over refined groups
# -------------------------------------------------------------------
# This version supersedes the prior aggressive attempt.
# It merges only within tight intent blocks and only when representative
# queries are very similar under a second-stage similarity threshold.

import ast
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

print("\n" + "=" * 80)
print("AGGRESSIVE PASS V2 - Start")
print("=" * 80)

if "prepared_df_refined" not in globals():
    raise RuntimeError("Run the refinement cell first to create `prepared_df_refined`.")

ref_df = prepared_df_refined.copy()
print(f"Input refined groups: {len(ref_df)}")

ACTION_PATTERNS_V2 = [
    (r"\b(list|show|what|which)\b", "ASK_LIST"),
    (r"\b(count|how many)\b", "ASK_COUNT"),
    (r"\b(forecast|predict|probability)\b", "ASK_PREDICT"),
    (r"\b(anomal|detect)\b", "ASK_ANOMALY"),
    (r"\b(failure mode)\b", "ASK_FAILURE_MODE"),
    (r"\b(work order|workorder)\b", "ASK_WORKORDER"),
    (r"\b(create|open|generate)\b", "ASK_CREATE"),
]
PARAM_SENSITIVE_PATTERNS_V2 = [
    r"\bcontext length\b",
    r"\btop\s+\d+\b",
    r"\bthreshold\b",
    r"\bwindow\b",
    r"\bforecast\b",
    r"\bpredict\b",
]


def to_list(v):
    if isinstance(v, list):
        return v
    if isinstance(v, str):
        try:
            p = ast.literal_eval(v)
            if isinstance(p, list):
                return p
            return [v]
        except Exception:
            return [v]
    return [v]


def detect_action_v2(query: str) -> str:
    q = str(query).lower()
    tags = [tag for pat, tag in ACTION_PATTERNS_V2 if re.search(pat, q)]
    if not tags:
        return "ASK_OTHER"
    return "+".join(sorted(set(tags)))


def is_param_sensitive_v2(query: str) -> bool:
    q = str(query).lower()
    return any(re.search(p, q) for p in PARAM_SENSITIVE_PATTERNS_V2)


ref_df["action_signature"] = ref_df["representative_query"].apply(detect_action_v2)
ref_df["param_sensitive"] = ref_df["representative_query"].apply(is_param_sensitive_v2)

print("\nSignature snapshot:")
print(ref_df[["cluster_id", "type", "entity", "action_signature", "param_sensitive", "representative_query"]].head(10).to_string(index=False))


def connected_components(sim_matrix, threshold):
    n = sim_matrix.shape[0]
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]
            x = parent[x]
        return x

    def union(a, b):
        ra, rb = find(a), find(b)
        if ra != rb:
            parent[rb] = ra

    for i in range(n):
        for j in range(i + 1, n):
            if sim_matrix[i, j] >= threshold:
                union(i, j)

    groups = {}
    for i in range(n):
        groups.setdefault(find(i), []).append(i)
    return list(groups.values())


def build_aggressive_v2(df_in, second_stage_threshold):
    rows = []
    meta_id = 0

    # Tight block: preserve metadata + inferred action + param-sensitive split
    block_cols = ["type", "entity", "category", "group", "action_signature", "param_sensitive"]

    for block_key, block in df_in.groupby(block_cols, dropna=False):
        block = block.copy().reset_index(drop=True)

        # Safety: do not merge parameter-sensitive groups
        if bool(block.iloc[0]["param_sensitive"]) or len(block) == 1:
            for _, r in block.iterrows():
                meta_id += 1
                rows.append({
                    "meta_cluster_id": meta_id,
                    "type": r["type"],
                    "entity": r["entity"],
                    "category": r["category"],
                    "group": r["group"],
                    "representative_query": r["representative_query"],
                    "representative_query_masked": r["representative_query_masked"],
                    "representative_expected_answer": r["representative_expected_answer"],
                    "representative_note": r["representative_note"],
                    "source_refined_cluster_ids": [r["cluster_id"]],
                    "source_refined_cluster_count": 1,
                    "cluster_size": int(r["cluster_size"]),
                    "member_queries": to_list(r["member_queries"]),
                    "member_expected_answers": to_list(r["member_expected_answers"]),
                    "member_notes": to_list(r["member_notes"]),
                    "action_signature": r["action_signature"],
                    "param_sensitive": bool(r["param_sensitive"]),
                    "merge_profile": f"aggressive_v2_threshold_{second_stage_threshold}",
                })
            continue

        # Non-param-sensitive: second-stage similarity over representative masked queries
        vec = TfidfVectorizer(ngram_range=(1, 2), min_df=1)
        X = vec.fit_transform(block["representative_query_masked"].astype(str).tolist())
        sim = cosine_similarity(X)
        comps = connected_components(sim, second_stage_threshold)

        for comp in comps:
            comp_block = block.iloc[comp].copy()

            # representative: largest support then shortest query
            comp_block = comp_block.sort_values(["cluster_size", "representative_query"], ascending=[False, True])
            rep = comp_block.iloc[0]

            merged_queries, merged_expected, merged_notes = [], [], []
            for _, rr in comp_block.iterrows():
                merged_queries.extend(to_list(rr["member_queries"]))
                merged_expected.extend(to_list(rr["member_expected_answers"]))
                merged_notes.extend(to_list(rr["member_notes"]))

            def dedup_keep_order(items):
                seen = set()
                out = []
                for x in items:
                    k = str(x)
                    if k not in seen:
                        seen.add(k)
                        out.append(x)
                return out

            merged_queries = dedup_keep_order(merged_queries)
            merged_expected = dedup_keep_order(merged_expected)
            merged_notes = dedup_keep_order(merged_notes)

            meta_id += 1
            rows.append({
                "meta_cluster_id": meta_id,
                "type": rep["type"],
                "entity": rep["entity"],
                "category": rep["category"],
                "group": rep["group"],
                "representative_query": rep["representative_query"],
                "representative_query_masked": rep["representative_query_masked"],
                "representative_expected_answer": rep["representative_expected_answer"],
                "representative_note": rep["representative_note"],
                "source_refined_cluster_ids": comp_block["cluster_id"].tolist(),
                "source_refined_cluster_count": int(len(comp_block)),
                "cluster_size": int(len(merged_queries)),
                "member_queries": merged_queries,
                "member_expected_answers": merged_expected,
                "member_notes": merged_notes,
                "action_signature": rep["action_signature"],
                "param_sensitive": bool(rep["param_sensitive"]),
                "merge_profile": f"aggressive_v2_threshold_{second_stage_threshold}",
            })

    return pd.DataFrame(rows)


# sweep second-stage thresholds
second_stage_thresholds = [0.75, 0.80, 0.83, 0.85]
sweep_rows = []
candidate_tables = {}

for t in second_stage_thresholds:
    tbl = build_aggressive_v2(ref_df, second_stage_threshold=t)
    candidate_tables[t] = tbl
    sweep_rows.append({
        "second_stage_threshold": t,
        "meta_groups": int(len(tbl)),
        "extra_compression_over_refined": float(len(ref_df) / max(len(tbl), 1)),
        "merged_meta_groups": int((tbl["source_refined_cluster_count"] > 1).sum()),
        "max_meta_cluster_size": int(tbl["cluster_size"].max()),
    })

sweep_df = pd.DataFrame(sweep_rows).sort_values("meta_groups").reset_index(drop=True)
print("\n" + "=" * 80)
print("AGGRESSIVE PASS V2 - Threshold sweep")
print("=" * 80)
print(sweep_df.to_string(index=False))

best_t = float(sweep_df.iloc[0]["second_stage_threshold"])
prepared_df_aggressive_v2 = candidate_tables[best_t].sort_values(["type", "entity", "meta_cluster_id"]).reset_index(drop=True)

print(f"\nChosen second-stage threshold: {best_t}")
print(f"Aggressive V2 groups: {len(prepared_df_aggressive_v2)}")
print(f"Extra compression over refined: {len(ref_df)/max(len(prepared_df_aggressive_v2),1):.3f}x")

print("\nLargest merged meta-groups:")
print(
    prepared_df_aggressive_v2.sort_values("cluster_size", ascending=False)
    [["meta_cluster_id", "type", "entity", "cluster_size", "source_refined_cluster_count", "action_signature", "param_sensitive", "representative_query"]]
    .head(10)
    .to_string(index=False)
)

# Export
out_dir = Path("data")
out_dir.mkdir(parents=True, exist_ok=True)

out_main = out_dir / "assetopsbench_skill_prep_representative_queries_aggressive_v2.csv"
out_skill = out_dir / "assetopsbench_skill_prep_for_skillgen_aggressive_v2.csv"
out_sweep = out_dir / "assetopsbench_skill_prep_aggressive_v2_sweep.csv"

prepared_df_aggressive_v2.to_csv(out_main, index=False)

skill_cols = [
    "meta_cluster_id", "type", "entity", "category", "group",
    "representative_query", "representative_expected_answer", "representative_note",
    "cluster_size", "member_queries", "member_expected_answers", "member_notes",
    "source_refined_cluster_ids", "source_refined_cluster_count",
    "action_signature", "param_sensitive", "merge_profile"
]
prepared_df_aggressive_v2[skill_cols].to_csv(out_skill, index=False)
sweep_df.to_csv(out_sweep, index=False)

print("\n" + "=" * 80)
print("AGGRESSIVE PASS V2 - Export")
print("=" * 80)
print(f"Aggressive V2 full CSV: {out_main.resolve()}")
print(f"Aggressive V2 skill CSV: {out_skill.resolve()}")
print(f"Aggressive V2 sweep CSV: {out_sweep.resolve()}")
print(f"Rows written: {len(prepared_df_aggressive_v2)}")

prepared_df_aggressive_v2.head(10)